## How to run this notebook

This notebook reads keyframes from Google Cloud Storage, runs OCR with PaddleOCR detection and VietOCR recognition, and uploads OCR annotation JSON shards back to GCS.

1. Enable GPU in Kaggle if available.
2. Create Kaggle Secrets named `GCS_BUCKET` and `GCS_SERVICE_ACCOUNT_JSON`.
3. Run `Install dependencies`; if Kaggle asks for a restart after installing PaddleOCR/PaddlePaddle, restart and continue from `Parameters`.
4. Edit the `Parameters` cell for prefixes, optional `SHOT_SEGMENTS_URI`, filters, batch sizes, workers, `OCR_DET_BATCH_SIZE`, and `OCR_RECOG_BATCH_SIZE`. Do not paste credentials into the notebook.
5. Run `Dry run` to confirm the source frames.
6. Run `Demo one batch` to inspect OCR text, bounding boxes, speed, and uploaded JSON.
7. When the demo output is correct, set `RUN_FULL = True` in the `Full run` cell and run the full job.
8. Outputs are written to `gs://<GCS_BUCKET>/<OUTPUT_PREFIX>/<run_id>/annotations/*.json` and `manifest.json`.


# FE OCR v1

Note: This Kaggle notebook reads frames from GCS, extracts one feature type, uploads JSON shards to GCS, and leaves Supabase formatting for a later step.


## Install dependencies

Note: Run this first on Kaggle. Restart the kernel if a package installer asks for it, especially in the OCR notebook.


In [ ]:
!pip install -q google-cloud-storage tqdm pandas numpy pillow opencv-python-headless vietocr
!pip install -q paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr


## Parameters

Note: Change only this cell for bucket names, prefixes, filters, batch sizes, workers, model settings, and output folders.


In [ ]:
from pathlib import Path
import os

TASK_NAME = "ocr"
KAGGLE_SECRET_GCS_BUCKET = "GCS_BUCKET"
KAGGLE_SECRET_GCS_SERVICE_ACCOUNT_JSON = "GCS_SERVICE_ACCOUNT_JSON"


def read_kaggle_secret(secret_name: str, default: str = "") -> str:
    """Read a Kaggle Secret, with an environment variable fallback for local testing."""
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(secret_name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(secret_name, default)


GCS_BUCKET = read_kaggle_secret(KAGGLE_SECRET_GCS_BUCKET)
GCS_SERVICE_ACCOUNT_JSON = read_kaggle_secret(KAGGLE_SECRET_GCS_SERVICE_ACCOUNT_JSON)
GCS_SERVICE_ACCOUNT_JSON_PATH = ""  # Optional local fallback path outside Kaggle.
FRAME_PREFIX = "processed/keyframes"
OUTPUT_PREFIX = "processed/feature-annotations/fe-ocr-v1"
GCS_PUBLIC_BASE_URL = ""
SHOT_SEGMENTS_URI = ""
DEFAULT_FPS = 25.0  # Used to estimate frame_seconds when shot_segments.csv is absent.

DATASET_CODE = "aic-2026"
DATASET_VERSION = "v1"
VIDEO_IDS = []
MAX_FRAMES = 0

PIPELINE_BATCH_SIZE = 16
DEMO_BATCH_SIZE = 4
DEMO_BATCH_INDEX = 0
DOWNLOAD_WORKERS = 16
GCS_TIMEOUT = 60
OUTPUT_SHARD_SIZE = 256
REUSE_LOCAL_DOWNLOADS = True
DELETE_LOCAL_AFTER_BATCH = True
CONTINUE_ON_BATCH_ERROR = True
DRY_RUN_SAMPLE = 10

OCR_MODEL_VERSION = "paddle-ppocrv5-mobile-det-vietocr-vgg-seq2seq-v1"
DETECTOR_MODEL = "PP-OCRv5_mobile_det"
DETECTOR_LIMIT_SIDE_LEN = 960
DETECTOR_LIMIT_TYPE = "max"
RECOGNIZER_MODEL = "vgg_seq2seq"
RECOGNIZER_WEIGHTS_URL = "https://vocr.vn/data/vietocr/vgg_seq2seq.pth"
RECOGNIZER_WEIGHTS_PATH = "/kaggle/working/model-cache/vietocr/vgg_seq2seq.pth"
OCR_DET_BATCH_SIZE = 8
OCR_RECOG_BATCH_SIZE = 32
LINE_Y_THRESHOLD = 35
LINE_X_GAP_THRESHOLD = 180
CROP_PADDING = 12
MODEL_CACHE_DIR = "/kaggle/working/model-cache"

LOCAL_FRAME_DIR = Path("/kaggle/working/frames-ocr")
LOCAL_OUTPUT_DIR = Path("/kaggle/working/fe-ocr-v1")


## Imports and runtime setup

Note: This cell loads common libraries, configures logging, and records the start timestamp.


In [ ]:
from __future__ import annotations

import csv
import json
import logging
import re
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path, PurePosixPath
from typing import Any, Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s - %(message)s")
LOGGER = logging.getLogger(TASK_NAME)


def utc_now() -> str:
    """Return the current UTC timestamp as an ISO-8601 string."""
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def make_run_id(task_name: str) -> str:
    """Create a unique run id for local and GCS output folders."""
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    return f"{task_name}-{stamp}-{uuid.uuid4().hex[:8]}"


print("Task:", TASK_NAME)
print("Started:", utc_now())


## GCS and output helpers

Note: These helpers list frames, download each batch with CPU workers, write JSON shards, upload to GCS, and print progress.


In [ ]:
FRAME_IDX_RE = re.compile(r"f(\d+)", re.IGNORECASE)
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}


@dataclass(frozen=True)
class FrameItem:
    """Metadata for one keyframe image stored in GCS."""

    bucket: str
    blob_name: str
    video_id: str
    image_name: str
    frame_idx: int
    frame_seconds: float
    fps: float | None = None
    shot_index: int | None = None
    frame_type: str | None = None

    @property
    def keyframe_id(self) -> str:
        """Return the backend-compatible keyframe id."""
        return f"{self.video_id}_F{self.frame_idx:06d}"

    @property
    def shot_id(self) -> str | None:
        """Return the backend-compatible shot id when shot metadata exists."""
        if self.shot_index is None:
            return None
        return f"{self.video_id}_S{self.shot_index:04d}"

    @property
    def gcs_uri(self) -> str:
        """Return the gs:// URI for this frame."""
        return f"gs://{self.bucket}/{self.blob_name}"


def make_storage_client():
    """Create a GCS client from Kaggle Secret JSON, local JSON path, or default auth."""
    from google.cloud import storage
    from google.oauth2 import service_account

    if not GCS_BUCKET:
        raise ValueError("Missing Kaggle Secret `GCS_BUCKET` or environment variable GCS_BUCKET.")

    raw_json = (GCS_SERVICE_ACCOUNT_JSON or "").strip()
    if raw_json:
        if raw_json.startswith("{"):
            info = json.loads(raw_json)
            credentials = service_account.Credentials.from_service_account_info(info)
            return storage.Client(credentials=credentials, project=info.get("project_id"))
        if Path(raw_json).exists():
            return storage.Client.from_service_account_json(raw_json)

    if GCS_SERVICE_ACCOUNT_JSON_PATH and Path(GCS_SERVICE_ACCOUNT_JSON_PATH).exists():
        return storage.Client.from_service_account_json(GCS_SERVICE_ACCOUNT_JSON_PATH)

    return storage.Client()



def parse_gcs_uri(uri: str) -> tuple[str, str]:
    """Split a gs://bucket/path URI into bucket and blob path."""
    if not uri.startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got {uri}")
    bucket, blob = uri[len("gs://") :].split("/", 1)
    return bucket, blob


def read_text(client, uri_or_path: str) -> str:
    """Read text from a local path or gs:// URI."""
    if uri_or_path.startswith("gs://"):
        bucket, blob = parse_gcs_uri(uri_or_path)
        return client.bucket(bucket).blob(blob).download_as_text(timeout=GCS_TIMEOUT)
    return Path(uri_or_path).read_text(encoding="utf-8-sig")


def normalize_video_folder(raw: str) -> str:
    """Convert video_id=L21_V001 folder names to L21_V001."""
    if raw.startswith("video_id="):
        return raw.split("=", 1)[1]
    return raw


def extract_frame_idx(image_name: str) -> int | None:
    """Extract an integer frame index from filenames containing f000123."""
    match = FRAME_IDX_RE.search(image_name)
    return int(match.group(1)) if match else None


def to_float(raw: object, default: float) -> float:
    """Convert a value to float, returning a default when it is missing."""
    if raw in (None, ""):
        return default
    return float(raw)


def to_optional_float(raw: object) -> float | None:
    """Convert a value to float or None when missing."""
    if raw in (None, ""):
        return None
    return float(raw)


def to_optional_int(raw: object) -> int | None:
    """Convert a value to int or None when missing."""
    if raw in (None, ""):
        return None
    return int(float(raw))


def load_shot_metadata(client) -> dict[tuple[str, str], dict[str, str]]:
    """Load optional shot_segments.csv rows keyed by (video_id, image_name)."""
    if not SHOT_SEGMENTS_URI:
        return {}
    rows: dict[tuple[str, str], dict[str, str]] = {}
    for row in csv.DictReader(StringIO(read_text(client, SHOT_SEGMENTS_URI))):
        image_path = str(row.get("image_path") or row.get("image_name") or "")
        image_name = PurePosixPath(image_path).name
        video_id = str(row.get("video_id") or PurePosixPath(image_path).parent.name or "").strip()
        if video_id and image_name:
            rows[(video_id, image_name)] = dict(row)
    return rows


def list_frames(client) -> list[FrameItem]:
    """List keyframe images from GCS, optionally filtered by VIDEO_IDS and MAX_FRAMES."""
    from google.api_core.retry import Retry

    selected_videos = set(VIDEO_IDS or []) or None
    metadata = load_shot_metadata(client)
    retry = Retry(initial=1.0, maximum=4.0, multiplier=2.0, deadline=GCS_TIMEOUT)
    frames: list[FrameItem] = []
    for blob in client.list_blobs(GCS_BUCKET, prefix=FRAME_PREFIX.strip("/"), timeout=GCS_TIMEOUT, retry=retry):
        path = PurePosixPath(blob.name)
        if path.suffix.lower() not in IMAGE_SUFFIXES or len(path.parts) < 2:
            continue
        video_id = normalize_video_folder(path.parts[-2])
        if selected_videos and video_id not in selected_videos:
            continue
        frame_idx = extract_frame_idx(path.name)
        if frame_idx is None:
            continue
        row = metadata.get((video_id, path.name), {})
        fps = to_optional_float(row.get("fps")) or DEFAULT_FPS
        frame_seconds = to_float(row.get("frame_sec"), frame_idx / fps if fps else 0.0)
        frames.append(
            FrameItem(
                bucket=GCS_BUCKET,
                blob_name=blob.name,
                video_id=video_id,
                image_name=path.name,
                frame_idx=frame_idx,
                frame_seconds=frame_seconds,
                fps=fps,
                shot_index=to_optional_int(row.get("shot_id")),
                frame_type=str(row.get("frame_type") or "").strip() or None,
            )
        )
        if MAX_FRAMES and len(frames) >= MAX_FRAMES:
            break
    frames = sorted(frames, key=lambda item: (item.video_id, item.frame_idx, item.image_name))
    LOGGER.info("Found %s frames across %s videos", len(frames), len({item.video_id for item in frames}))
    return frames


def chunked(items: Iterable[Any], size: int) -> Iterable[list[Any]]:
    """Yield fixed-size batches from an iterable."""
    batch: list[Any] = []
    for item in items:
        batch.append(item)
        if len(batch) >= size:
            yield batch
            batch = []
    if batch:
        yield batch


def download_frame_batch(client, frames: list[FrameItem], local_root: Path) -> tuple[list[Path], float]:
    """Download one batch of frames from GCS using DOWNLOAD_WORKERS threads."""
    started = time.perf_counter()
    bucket = client.bucket(GCS_BUCKET)
    local_paths = [local_root / item.video_id / item.image_name for item in frames]

    def download_one(item: FrameItem, destination: Path) -> Path:
        """Download one image unless a reusable local copy exists."""
        destination.parent.mkdir(parents=True, exist_ok=True)
        if REUSE_LOCAL_DOWNLOADS and destination.exists() and destination.stat().st_size > 0:
            return destination
        bucket.blob(item.blob_name).download_to_filename(str(destination), timeout=GCS_TIMEOUT)
        return destination

    with ThreadPoolExecutor(max_workers=max(1, DOWNLOAD_WORKERS)) as pool:
        futures = [pool.submit(download_one, item, path) for item, path in zip(frames, local_paths)]
        for future in as_completed(futures):
            future.result()
    return local_paths, time.perf_counter() - started


def public_url(blob_name: str) -> str:
    """Build a public URL for a GCS object."""
    if GCS_PUBLIC_BASE_URL:
        return f"{GCS_PUBLIC_BASE_URL.rstrip('/')}/{blob_name}"
    return f"https://storage.googleapis.com/{GCS_BUCKET}/{blob_name}"


def base_frame_payload(item: FrameItem, model_name: str, model_version: str) -> dict[str, Any]:
    """Build common JSON fields aligned with keyframes/frame_annotations tables."""
    return {
        "schema_version": "feature-extraction-v1",
        "task": TASK_NAME,
        "dataset": {"code": DATASET_CODE, "version": DATASET_VERSION},
        "video_id": item.video_id,
        "keyframe_id": item.keyframe_id,
        "shot_id": item.shot_id,
        "shot_index": item.shot_index,
        "frame_idx": item.frame_idx,
        "frame_seconds": item.frame_seconds,
        "timestamp_ms": int(item.frame_seconds * 1000),
        "frame_type": item.frame_type or "key",
        "image_name": item.image_name,
        "image_rel_path": f"{item.video_id}/{item.image_name}",
        "image_storage_key": item.blob_name,
        "image_uri": item.gcs_uri,
        "image_url": public_url(item.blob_name),
        "model_name": model_name,
        "model_version": model_version,
        "created_at": utc_now(),
    }


def upload_json_text(client, gcs_path: str, payload: str) -> str:
    """Upload JSON text to GCS and return the uploaded gs:// URI."""
    client.bucket(GCS_BUCKET).blob(gcs_path).upload_from_string(payload, content_type="application/json")
    return f"gs://{GCS_BUCKET}/{gcs_path}"


def save_and_upload_part(client, records: list[dict[str, Any]], run_id: str, part_index: int) -> dict[str, Any]:
    """Save one JSON array shard locally and upload it to GCS."""
    out_dir = LOCAL_OUTPUT_DIR / run_id / "annotations"
    out_dir.mkdir(parents=True, exist_ok=True)
    filename = f"part-{part_index:06d}.json"
    payload = json.dumps(records, ensure_ascii=False)
    local_path = out_dir / filename
    local_path.write_text(payload, encoding="utf-8")
    started = time.perf_counter()
    gcs_uri = upload_json_text(client, f"{OUTPUT_PREFIX.strip('/')}/{run_id}/annotations/{filename}", payload)
    return {
        "part_index": part_index,
        "records": len(records),
        "local_path": str(local_path),
        "gcs_uri": gcs_uri,
        "upload_seconds": round(time.perf_counter() - started, 3),
    }


def write_manifest(client, run_id: str, manifest: dict[str, Any]) -> str:
    """Write and upload the run manifest."""
    out_dir = LOCAL_OUTPUT_DIR / run_id
    out_dir.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(manifest, ensure_ascii=False, indent=2)
    (out_dir / "manifest.json").write_text(payload, encoding="utf-8")
    return upload_json_text(client, f"{OUTPUT_PREFIX.strip('/')}/{run_id}/manifest.json", payload)


def remove_local_files(paths: list[Path]) -> None:
    """Delete downloaded frame files after each batch when configured."""
    if not DELETE_LOCAL_AFTER_BATCH:
        return
    for path in paths:
        path.unlink(missing_ok=True)


def log_batch(batch_index: int, batch_count: int, batch_size: int, processed: int, total: int, timings: dict[str, float], uploaded: dict[str, Any] | None) -> None:
    """Print progress, percentage, timing, throughput, and upload URI for one batch."""
    percent = 100.0 * processed / max(total, 1)
    batch_seconds = timings.get("batch_seconds", 0.0)
    speed = batch_size / max(batch_seconds, 1e-9)
    upload_uri = uploaded["gcs_uri"] if uploaded else "pending"
    print(
        f"[{TASK_NAME}] batch {batch_index}/{batch_count} | "
        f"frames={batch_size} processed={processed}/{total} ({percent:.2f}%) | "
        f"download={timings.get('download_seconds', 0):.2f}s "
        f"inference={timings.get('inference_seconds', 0):.2f}s "
        f"upload={timings.get('upload_seconds', 0):.2f}s "
        f"batch={batch_seconds:.2f}s | speed={speed:.2f} frames/s | upload={upload_uri}"
    )


## Model and extraction functions

Note: This cell is task-specific. It loads the model lazily and turns one local image batch into JSON annotation records.


In [ ]:
import urllib.request
from collections import defaultdict

import cv2
import torch
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OCR_DETECTOR = None
OCR_PREDICTOR = None


def download_file(url: str, destination: Path) -> Path:
    """Download a model file with a progress bar when it is missing."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    tmp_path = destination.with_suffix(destination.suffix + ".tmp")
    with tqdm(unit="B", unit_scale=True, desc=f"Downloading {destination.name}") as progress:
        def reporthook(blocks: int, block_size: int, total_size: int) -> None:
            """Update the tqdm bar from urllib callbacks."""
            if total_size > 0:
                progress.total = total_size
            progress.update(max(0, blocks * block_size - progress.n))
        urllib.request.urlretrieve(url, tmp_path, reporthook)
    tmp_path.replace(destination)
    return destination


def load_model() -> None:
    """Load PaddleOCR text detector and VietOCR recognizer once."""
    global OCR_DETECTOR, OCR_PREDICTOR
    if OCR_DETECTOR is not None:
        return
    from paddleocr import TextDetection
    from vietocr.tool.config import Cfg
    from vietocr.tool.predictor import Predictor

    weights = download_file(RECOGNIZER_WEIGHTS_URL, Path(RECOGNIZER_WEIGHTS_PATH))
    config = Cfg.load_config_from_name(RECOGNIZER_MODEL)
    config["cnn"]["pretrained"] = False
    config["device"] = DEVICE
    config["weights"] = str(weights)
    OCR_PREDICTOR = Predictor(config)
    OCR_DETECTOR = TextDetection(model_name=DETECTOR_MODEL, device="gpu" if DEVICE == "cuda" else "cpu", limit_side_len=DETECTOR_LIMIT_SIDE_LEN, limit_type=DETECTOR_LIMIT_TYPE)
    print("Loaded OCR:", DETECTOR_MODEL, RECOGNIZER_MODEL, "device:", DEVICE)


def clear_gpu_cache() -> None:
    """Release CUDA cache after a failed batch."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def poly_to_xyxy(poly: np.ndarray) -> list[float]:
    """Convert a four-point polygon to x1, y1, x2, y2."""
    return [float(np.min(poly[:, 0])), float(np.min(poly[:, 1])), float(np.max(poly[:, 0])), float(np.max(poly[:, 1]))]


def merge_boxes_by_line(polys: list[np.ndarray]) -> list[list[float]]:
    """Merge OCR polygons that appear on the same text line."""
    boxes = sorted([poly_to_xyxy(poly) for poly in polys], key=lambda box: (box[1], box[0]))
    lines: list[dict[str, list[float]]] = []
    for box in boxes:
        x1, y1, x2, y2 = box
        cy = (y1 + y2) / 2
        for line in lines:
            lx1, ly1, lx2, ly2 = line["box"]
            lcy = (ly1 + ly2) / 2
            if abs(cy - lcy) < LINE_Y_THRESHOLD and x1 - lx2 < LINE_X_GAP_THRESHOLD:
                line["box"] = [min(lx1, x1), min(ly1, y1), max(lx2, x2), max(ly2, y2)]
                break
        else:
            lines.append({"box": box})
    return [line["box"] for line in lines]


def crop_xyxy(rgb: np.ndarray, box: list[float]) -> np.ndarray:
    """Crop a padded x1, y1, x2, y2 region from an RGB image."""
    h, w = rgb.shape[:2]
    x1, y1, x2, y2 = map(int, box)
    return rgb[max(0, y1 - CROP_PADDING):min(h, y2 + CROP_PADDING), max(0, x1 - CROP_PADDING):min(w, x2 + CROP_PADDING)]


def extract_polys(det_result: object) -> list[np.ndarray]:
    """Extract PaddleOCR detection polygons from a result object."""
    boxes = det_result.get("dt_polys", []) if isinstance(det_result, dict) else getattr(det_result, "dt_polys", [])
    return [np.asarray(box, dtype=np.float32) for box in boxes if np.asarray(box).shape == (4, 2)]


def predict_detector(paths: list[Path]) -> list[Any]:
    """Run text detection on a path batch with one-by-one fallback."""
    try:
        return list(OCR_DETECTOR.predict([str(path) for path in paths]))
    except Exception:
        outputs = []
        for path in paths:
            outputs.extend(list(OCR_DETECTOR.predict(str(path))))
        return outputs


def recognize_crops(crops: list[Image.Image]) -> list[str]:
    """Recognize text in crop images using VietOCR batch mode when available."""
    if not crops:
        return []
    if hasattr(OCR_PREDICTOR, "predict_batch"):
        texts = OCR_PREDICTOR.predict_batch(crops, batch_size=OCR_RECOG_BATCH_SIZE)
    else:
        texts = [OCR_PREDICTOR.predict(crop) for crop in crops]
    return [str(text).strip() for text in texts]


def extract_ocr(local_paths: list[Path]) -> list[dict[str, Any]]:
    """Extract OCR texts and regions from one local image batch."""
    load_model()
    det_results = []
    for path_batch in chunked(local_paths, OCR_DET_BATCH_SIZE):
        det_results.extend(predict_detector(path_batch))
    outputs = [{"texts": [], "regions": []} for _ in local_paths]
    crops: list[Image.Image] = []
    refs: list[tuple[int, list[float]]] = []
    for image_i, (path, det_result) in enumerate(zip(local_paths, det_results)):
        img = cv2.imread(str(path))
        if img is None:
            continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        for box in merge_boxes_by_line(extract_polys(det_result)):
            crop = crop_xyxy(rgb, box)
            if crop is not None and crop.shape[0] >= 5 and crop.shape[1] >= 5:
                crops.append(Image.fromarray(crop))
                refs.append((image_i, box))
    for (image_i, box), text in zip(refs, recognize_crops(crops)):
        if text:
            outputs[image_i]["texts"].append(text)
            outputs[image_i]["regions"].append({"text": text, "bbox_xyxy": [round(float(v), 2) for v in box]})
    return outputs


def extract_records(frames: list[FrameItem], local_paths: list[Path]) -> list[dict[str, Any]]:
    """Create OCR annotation JSON records for one frame batch."""
    outputs = extract_ocr(local_paths)
    records = []
    for item, output in zip(frames, outputs):
        texts = output["texts"]
        record = base_frame_payload(item, DETECTOR_MODEL, OCR_MODEL_VERSION)
        record["annotation"] = {"kind": "OCR", "text_value": " ".join(texts).strip(), "ocr_texts": texts, "json_value": output, "confidence": 1.0, "model_version": OCR_MODEL_VERSION}
        records.append(record)
    return records


def model_metadata() -> dict[str, Any]:
    """Return OCR model metadata for the manifest."""
    return {"detector_model": DETECTOR_MODEL, "recognizer_model": RECOGNIZER_MODEL, "model_version": OCR_MODEL_VERSION, "device": DEVICE}


def runtime_parameters() -> dict[str, Any]:
    """Return tunable OCR parameters for the manifest."""
    return {"pipeline_batch_size": PIPELINE_BATCH_SIZE, "download_workers": DOWNLOAD_WORKERS, "ocr_det_batch_size": OCR_DET_BATCH_SIZE, "ocr_recog_batch_size": OCR_RECOG_BATCH_SIZE, "output_shard_size": OUTPUT_SHARD_SIZE}


## Run helpers

Note: These helpers implement dry run, demo one batch, and full extraction with timing metrics.


In [ ]:
def run_dry_run() -> dict[str, Any]:
    """List frames and print a sample without downloading images or loading models."""
    client = make_storage_client()
    frames = list_frames(client)
    report = {
        "task": TASK_NAME,
        "mode": "dry_run",
        "frames": len(frames),
        "videos": len({item.video_id for item in frames}),
        "sample": [asdict(item) | {"keyframe_id": item.keyframe_id, "gcs_uri": item.gcs_uri} for item in frames[:DRY_RUN_SAMPLE]],
        "created_at": utc_now(),
    }
    print(json.dumps(report, ensure_ascii=False, indent=2))
    return report


def run_extraction(mode: str, max_batches: int | None = None) -> dict[str, Any]:
    """Run demo or full extraction and upload JSON shards plus a manifest to GCS."""
    client = make_storage_client()
    frames = list_frames(client)
    if mode == "demo":
        start = DEMO_BATCH_INDEX * DEMO_BATCH_SIZE
        frames = frames[start : start + DEMO_BATCH_SIZE]
    if not frames:
        raise RuntimeError("No frames found. Check GCS_BUCKET, FRAME_PREFIX, VIDEO_IDS, and credentials.")

    run_id = make_run_id(TASK_NAME if mode == "full" else f"{TASK_NAME}-{mode}")
    local_frame_root = LOCAL_FRAME_DIR / run_id
    batches = list(chunked(frames, PIPELINE_BATCH_SIZE))
    if max_batches is not None:
        batches = batches[:max_batches]
    total_frames = sum(len(batch) for batch in batches)
    embedding_index_by_key: dict[str, int] = {}
    video_offsets: dict[str, int] = {}
    for item in frames:
        index_0 = video_offsets.get(item.video_id, 0)
        embedding_index_by_key[item.keyframe_id] = index_0
        video_offsets[item.video_id] = index_0 + 1

    print("Run id:", run_id)
    print("Mode:", mode, "frames:", total_frames, "batches:", len(batches))
    load_model()

    started = time.perf_counter()
    processed = 0
    failed = 0
    part_index = 1
    pending: list[dict[str, Any]] = []
    parts: list[dict[str, Any]] = []
    batch_metrics: list[dict[str, Any]] = []

    for batch_index, batch in enumerate(tqdm(batches, desc=f"{TASK_NAME} batches", unit="batch"), start=1):
        batch_started = time.perf_counter()
        uploaded = None
        local_paths: list[Path] = []
        try:
            local_paths, download_seconds = download_frame_batch(client, batch, local_frame_root)
            inference_started = time.perf_counter()
            records = extract_records(batch, local_paths)
            inference_seconds = time.perf_counter() - inference_started
            for record in records:
                record["run_id"] = run_id
                record["batch_index"] = batch_index
                record["embedding_index_0"] = embedding_index_by_key.get(record.get("keyframe_id"))
            pending.extend(records)
            processed += len(records)
            upload_seconds = 0.0
            if len(pending) >= OUTPUT_SHARD_SIZE or batch_index == len(batches):
                uploaded = save_and_upload_part(client, pending, run_id, part_index)
                parts.append(uploaded)
                upload_seconds = uploaded["upload_seconds"]
                part_index += 1
                pending = []
            timings = {
                "download_seconds": download_seconds,
                "inference_seconds": inference_seconds,
                "upload_seconds": upload_seconds,
                "batch_seconds": time.perf_counter() - batch_started,
            }
            batch_metrics.append({"batch_index": batch_index, "frames": len(batch), **{k: round(v, 3) for k, v in timings.items()}})
            log_batch(batch_index, len(batches), len(batch), processed, total_frames, timings, uploaded)
        except Exception as exc:
            failed += len(batch)
            LOGGER.exception("Batch %s failed: %s", batch_index, exc)
            clear_gpu_cache()
            if not CONTINUE_ON_BATCH_ERROR:
                raise
        finally:
            remove_local_files(local_paths)

    elapsed = time.perf_counter() - started
    manifest = {
        "schema_version": "feature-extraction-v1",
        "task": TASK_NAME,
        "mode": mode,
        "run_id": run_id,
        "created_at": utc_now(),
        "dataset": {"code": DATASET_CODE, "version": DATASET_VERSION},
        "input": {"bucket": GCS_BUCKET, "frame_prefix": FRAME_PREFIX, "shot_segments_uri": SHOT_SEGMENTS_URI, "video_ids": VIDEO_IDS, "max_frames": MAX_FRAMES},
        "model": model_metadata(),
        "parameters": runtime_parameters(),
        "totals": {"frames": total_frames, "processed": processed, "failed": failed, "parts": len(parts)},
        "timing": {"elapsed_seconds": round(elapsed, 3), "avg_frames_per_second": round(processed / max(elapsed, 1e-9), 3)},
        "output_parts": parts,
        "batch_metrics": batch_metrics,
        "supabase_schema_hint": {"frame_table": "keyframes", "annotation_table": "frame_annotations", "join_key": "keyframe_id/frame_id"},
    }
    manifest_uri = write_manifest(client, run_id, manifest)
    manifest["manifest_uri"] = manifest_uri
    print(json.dumps({"run_id": run_id, "manifest_uri": manifest_uri, "totals": manifest["totals"], "timing": manifest["timing"]}, ensure_ascii=False, indent=2))
    return manifest


def run_demo_batch() -> dict[str, Any]:
    """Run exactly one demo batch and upload its JSON output."""
    return run_extraction(mode="demo", max_batches=1)


def run_full() -> dict[str, Any]:
    """Run extraction over every configured frame."""
    return run_extraction(mode="full", max_batches=None)


## Dry run

Note: This cell only lists GCS frame metadata. It does not download frames, load models, or upload output.


In [ ]:
dry_report = run_dry_run()


## Demo one batch

Note: This cell processes one small batch, uploads one JSON shard, and writes a manifest. Run this before the full job.


In [ ]:
demo_report = run_demo_batch()


## Full run

Note: Set `RUN_FULL = True` only after dry run and demo are OK. This may run for hours.


In [ ]:
RUN_FULL = False

if RUN_FULL:
    full_report = run_full()
else:
    print("Set RUN_FULL = True in this cell to process all configured frames.")


## Output check

Note: This cell lists local manifest files created in the Kaggle working directory.


In [ ]:
for path in sorted(LOCAL_OUTPUT_DIR.rglob("manifest.json")):
    print(path)
